In [ ]:
import os
import sys

# janky hack, need to add install to .toml
module_path = os.path.abspath(os.path.join("..", "../ophir"))

# Add the path to sys.path
if module_path not in sys.path:
    sys.path.insert(0, module_path)


import lightning as L
import pandas as pd
import torch
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from ophir.coin_datasets import construct_datasets
from ophir.training_models import LightningMulitClassPricePredictor

torch.set_float32_matmul_precision("high")
torch._functorch.config.donated_buffer = False
from datetime import timedelta

In [ ]:
dir_path = "../data"
min_value = -10
max_value = 10
elements_per_sample = 180
elements_in_encoder = 150
train_loader, test_loader, sector_tokens, stock_tokens = construct_datasets(
    dir_path, "h", min_value, max_value, elements_per_sample, elements_in_encoder
)

In [ ]:
# Run this cell train new model
ckpt_path = "saved-models/"
model = LightningMulitClassPricePredictor(
    emb_dim=128,
    num_dec_layers=4,
    num_heads=8,
    min_value=min_value,
    max_value=max_value,
    elements_in_encoder=elements_in_encoder,
    num_sector=len(sector_tokens),
    num_stock=len(stock_tokens),
)

# uncomment this out to load a previous model
# model = LightningMulitClassPricePredictor.load_from_checkpoint(
#     "saved-models/ophir-small-stock-weights-modifiers.ckpt"
# )

In [ ]:
trainer = L.Trainer(
    precision="16-mixed",
    max_epochs=1000,
    default_root_dir=ckpt_path,
    accelerator="cuda",
    callbacks=[
        ModelCheckpoint(
            ckpt_path,
            filename="ophir-small-stock-weights-modifiers",
            train_time_interval=timedelta(seconds=60 * 5),
        ),
        LearningRateMonitor("step"),
    ],
    logger=TensorBoardLogger(ckpt_path, name="tensorboard-logger"),
)

trainer.fit(
    model=model,
    train_dataloaders=train_loader,
    val_dataloaders=test_loader,
    # ckpt_path="saved-models/ophir-small-stock-weights-modifiers.ckpt"  # uncomment this out if resume training
)